In [1]:
import os
import requests
import json
from typing import Optional, Dict, Any
import pandas as pd
from dotenv import load_dotenv

from utils import DuckdbUtils
from dags.data_collector import CryptoDataCollector

1. raw layer

In [2]:
sample_asset = 'BTC'

sample_data_collector = CryptoDataCollector(
    sample_asset,
    'iceberg.cryptocurrencies_project_raw.btc',
    'iceberg.cryptocurrencies_project_raw_stg.btc',
)

In [3]:
sample_result_row_cnt = sample_data_collector.process()

/opt/workspace/ETL/dags/data_collector.py:54: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pdf = pd.read_json(json_data_str, orient='index')


In [4]:
print(f"Successfully loaded {sample_result_row_cnt} rows")

Successfully loaded 5714 rows


2. dds layer

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
os.environ['SPARK_CONF_DIR'] = '/opt/spark/conf'

from utils import PySparkTableLoader

In [2]:
spark = SparkSession \
    .builder \
    .appName("union_cryptoasset_data") \
    .master(os.getenv('SPARK_MASTER_URL')) \
    .config('spark.ui.port', '4040') \
    .config('spark.driver.cores', '1') \
    .config('spark.driver.memory', '1G') \
    .config('spark.executor.instances', '1') \
    .config('spark.executor.cores', '1') \
    .config('spark.executor.memory', '1G') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
spark.conf.set('spark.sql.shuffle.partitions', '10')

26/03/15 12:45:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
DDS_MAX_DATE = spark.sql("SELECT CAST(COALESCE(DATE_TRUNC('year', MAX(dt) - interval '7 days'), '2020-01-01') AS DATE) FROM iceberg.cryptocurrencies_project_dds.trade_data").collect()[0][0]

In [4]:
DDS_QUERY = f"""
    SELECT asset
        , dt
        , open_price
        , high_price
        , low_price
        , close_price
        , trading_volume
        , src_processed_dttm
        , processed_dttm
    FROM (
        SELECT *
            , ROW_NUMBER() OVER(PARTITION BY asset, dt ORDER BY src_processed_dttm DESC) AS rn
        FROM (
            SELECT 'BTC' AS asset
                    , dt
                    , open_price
                    , high_price
                    , low_price
                    , close_price
                    , trading_volume
                    , processed_dttm AS src_processed_dttm
                    , CURRENT_TIMESTAMP AS processed_dttm
            FROM iceberg.cryptocurrencies_project_raw.btc
            WHERE dt >= '{DDS_MAX_DATE}'
            UNION ALL
            SELECT 'ETH' AS asset
                    , dt
                    , open_price
                    , high_price
                    , low_price
                    , close_price
                    , trading_volume
                    , processed_dttm AS src_processed_dttm
                    , CURRENT_TIMESTAMP AS processed_dttm
            FROM iceberg.cryptocurrencies_project_raw.eth
            WHERE dt >= '{DDS_MAX_DATE}'
        ) AS united_data
    ) AS dedubled_data
    WHERE rn = 1
    ;
"""

In [5]:
dds_table_loader = PySparkTableLoader(
    table_name='iceberg.cryptocurrencies_project_dds.trade_data',
    stg_table_name='iceberg.cryptocurrencies_project_dds_stg.trade_data',
    partition_by=['asset', f.years('dt')],
    spark_session=spark,
)

In [6]:
dds_table_loader\
    .calc_stg(DDS_QUERY)\
    .load_trg_scd1(overwrite_trg=False)

3. dm layer

In [3]:
REPORT_DT = "current_date"

In [4]:
DM_QUERY = f"""
    SELECT asset
        , {REPORT_DT} AS report_dt
        , current_price
        , ROUND(100 * (2 - (high_price_exactly_7d + low_price_exactly_7d) / current_price) , 2) AS div_percent_7d
        , ROUND(100 * (2 - (high_price_exactly_30d + low_price_exactly_30d) / current_price) , 2) AS div_percent_30d
        , ROUND(100 * (2 - (high_price_exactly_90d + low_price_exactly_90d) / current_price) , 2) AS div_percent_90d
        , ROUND(100 * (2 - (high_price_exactly_1y + low_price_exactly_1y) / current_price) , 2) AS div_percent_1y
        , ROUND(100 * (high_price_7d / low_price_7d), 2) AS volatility_percent_7d
        , ROUND(100 * (high_price_30d / low_price_30d), 2) AS volatility_percent_30d
        , ROUND(100 * (high_price_90d / low_price_90d), 2) AS volatility_percent_90d
        , ROUND(100 * (high_price_1y / low_price_1y), 2) AS volatility_percent_1y
        , high_price_7d
        , high_price_30d
        , high_price_90d
        , high_price_1y   
        , low_price_7d
        , low_price_30d
        , low_price_90d
        , low_price_1y
        , med_price_7d
        , med_price_30d
        , med_price_90d
        , med_price_1y
        , src_processed_dttm
        , CURRENT_TIMESTAMP AS processed_dttm
    FROM (
        SELECT hist_data.asset
            , MAX(current_data.current_price) AS current_price
            , MAX(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 7 days AND {REPORT_DT}
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_7d
            , MAX(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 30 days AND {REPORT_DT}
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_30d
            , MAX(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 90 days AND {REPORT_DT}
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_90d
            , MAX(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 1 year AND {REPORT_DT}
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_1y
            , MIN(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 7 days AND {REPORT_DT}
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_7d
            , MIN(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 30 days AND {REPORT_DT}
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_30d
            , MIN(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 90 days AND {REPORT_DT}
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_90d
            , MIN(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 1 year AND {REPORT_DT}
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_1y
            , AVG(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 7 days AND {REPORT_DT}
                    THEN (hist_data.high_price + hist_data.low_price) / 2
                    ELSE 0
            END) AS med_price_7d
            , AVG(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 30 days AND {REPORT_DT}
                    THEN (hist_data.high_price + hist_data.low_price) / 2
                    ELSE 0
            END) AS med_price_30d
            , AVG(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 90 days AND {REPORT_DT}
                    THEN (hist_data.high_price + hist_data.low_price) / 2
                    ELSE 0
            END) AS med_price_90d
            , AVG(CASE
                WHEN hist_data.dt BETWEEN {REPORT_DT} - interval 1 year AND {REPORT_DT}
                    THEN (hist_data.high_price + hist_data.low_price) / 2
                    ELSE 0
            END) AS med_price_1y
            , MAX(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 7 days
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_exactly_7d
            , MAX(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 30 days
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_exactly_30d
            , MAX(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 90 days
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_exactly_90d
            , MAX(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 1 year
                    THEN hist_data.high_price
                    ELSE 0
            END) AS high_price_exactly_1y
            , MIN(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 7 days
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_exactly_7d
            , MIN(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 30 days
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_exactly_30d
            , MIN(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 90 days
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_exactly_90d
            , MIN(CASE
                WHEN hist_data.dt = {REPORT_DT} - interval 1 year
                    THEN hist_data.low_price
                    ELSE 0
            END) AS low_price_exactly_1y
            , MAX(hist_data.src_processed_dttm) AS src_processed_dttm
            , CURRENT_TIMESTAMP AS processed_dttm
        FROM iceberg.cryptocurrencies_project_dds.trade_data AS hist_data
        INNER JOIN (
            SELECT asset, MAX(low_price - high_price) / 2 AS current_price
            FROM iceberg.cryptocurrencies_project_dds.trade_data
            WHERE dt = {REPORT_DT}
            GROUP BY asset
        ) AS current_data
            ON hist_data.asset = current_data.asset
        WHERE hist_data.dt BETWEEN {REPORT_DT} - interval 1 year AND {REPORT_DT}
        GROUP BY hist_data.asset
    ) AS prep
    ;
"""

In [6]:
dm_table_loader = PySparkTableLoader(
    table_name='iceberg.cryptocurrencies_project_dm.analytic_indicators',
    stg_table_name='iceberg.cryptocurrencies_project_dm_stg.analytic_indicators',
    partition_by=['asset', 'report_dt'],
    spark_session=spark,
)

In [7]:
dm_table_loader \
    .calc_stg(DM_QUERY)\
    .load_trg_scd1(overwrite_trg=False)

In [4]:
spark.stop()

In [12]:
pd.set_option("max_colwidth", 1000)